In [1]:
# =================================================================
# 06_baseline_rf_model
# Goal: 
# 1. Load the unified dataset and IMMEDIATELY drop heavy 3D matrices.
# 2. Extract original basin IDs to prevent data leakage during split.
# 3. Train a Baseline Random Forest Classifier.
# 4. Evaluate performance and plot Feature Importance.
# =================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import gc # Garbage Collector to manually free up memory

# --- 1. CONFIGURATION ---
DATASET_PATH = r"D:\Development\RESEARCH\urban_flood_database\chronicle\unified_ml_dataset_V01.pkl"
URBAN_THRESHOLD = 20.0 
THRESH_1H = 7.0
THRESH_24H = 1.0

# --- 2. LOAD AND PREPARE DATA (MEMORY OPTIMIZED) ---
print("Loading dataset into memory...")
df = pd.read_pickle(DATASET_PATH)

print("Dropping heavy 3D matrices to free up RAM...")
# We do not need the raw pixel matrices for tabular Machine Learning
columns_to_drop = ['imerg_matrix', 'imerg_mask', 'imerg_meta', 'geometry_wkt']
df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

# Force Python to clear the unneeded heavy data from RAM immediately
gc.collect() 

print("Filtering dataset based on thresholds...")
filtered_df = df[
    (df['urban_percentage'] >= URBAN_THRESHOLD) &
    (df['60_max_rainfall_intens'] > THRESH_1H) &
    (df['1440_max_rainfall_intens'] > THRESH_24H)
].copy()

# Extract the ORIGINAL event ID for grouping
filtered_df['original_basin_id'] = filtered_df['event_id'].apply(lambda x: str(x).split('_noflood_')[0])

print(f"Total valid events for modeling: {len(filtered_df)}")

# --- 3. DEFINE FEATURES (X) AND TARGET (y) ---
feature_cols = [
    'area_km2', 'urban_percentage', 
    'upa_max', 'upa_p95', 'upa_p99', 
    'PFDI_max', 'PFDI_p95', 'PFDI_p99',
    '30_max_rainfall_intens', '60_max_rainfall_intens', 
    '120_max_rainfall_intens', '240_max_rainfall_intens', 
    '360_max_rainfall_intens', '720_max_rainfall_intens', 
    '1440_max_rainfall_intens'
]

feature_cols = [col for col in feature_cols if col in filtered_df.columns]

X = filtered_df[feature_cols]
y = filtered_df['is_flood']
groups = filtered_df['original_basin_id']

# --- 4. TRAIN/TEST SPLIT (GROUPED) ---
print("Splitting data into Train and Test sets (Grouped by Basin)...")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Training set size: {len(X_train)} | Testing set size: {len(X_test)}")

# --- 5. TRAIN RANDOM FOREST MODEL ---
print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# --- 6. EVALUATION ---
print("\nPredicting on Test Set...")
y_pred = rf_model.predict(X_test)

print("\n================ CLASSIFICATION REPORT ================")
print(classification_report(y_test, y_pred, target_names=['No Flood (0)', 'Flood (1)']))
print("=======================================================\n")

# --- 7. VISUALIZATIONS ---
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot A: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Flood', 'Flood'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontsize=14)
axes[0].grid(False)

# Plot B: Feature Importance
importances = rf_model.feature_importances_
feature_names = X.columns
feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False)

sns.barplot(data=feat_imp_df, x='Importance', y='Feature', ax=axes[1], palette='viridis')
axes[1].set_title('Random Forest - Feature Importance', fontsize=14)
axes[1].set_xlabel('Relative Importance')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

Loading dataset into memory...


SystemError: deallocated bytearray object has exported buffers

MemoryError: 